Since there is a retraining needed for poor accuracy values. So to hold up the FGP masks we need a interactive environment to retain and experiment with diffrent parameters.

In [1]:

import sys
import torch
from torch import nn
import copy
import random
import torch
from pathlib import Path

# Notebook is inside .../PruningNAS/PruningNAS, so add project root (.../PruningNAS)
project_root = Path.cwd().parent
if str(project_root) not in sys.path:
	sys.path.insert(0, str(project_root))

# Install local package in editable mode (run once; safe to re-run)
%pip install -e ..


from PruningNAS.DataProcess.DataPreprocessing import get_dataloaders
from PruningNAS.Utills.EvaluatiorUtills import get_model_size, get_sparsity
from PruningNAS.Utills.PrunUtillCP import ChannelPrunner
from PruningNAS.Utills.PrunUtillFGP import FineGrainedPruner
from PruningNAS.Utills.TrainingModulesUtills import TrainingPrunned, evaluate
from PruningNAS.Utills.Utill import print_model
from PruningNAS.Utills.ViewerUtills import plot_accuracy, plot_loss  # Ensure you import your correct model architecture
seed=0
random.seed(seed)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

Byte = 8
KiB = 1024 * Byte
MiB = 1024 * KiB
GiB = 1024 * MiB

Obtaining file:///D:/Sutanu_WorkSpace/PruningNas
Note: you may need to restart the kernel to use updated packages.
Device: cuda


ERROR: file:///D:/Sutanu_WorkSpace/PruningNas does not appear to be a Python project: neither 'setup.py' nor 'pyproject.toml' found.
d:\Sutanu_WorkSpace\PruningNAS\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Set Static Params here:

In [2]:
import os


current_dir = os.getcwd()
print(f"Current working directory: {current_dir}")


Current working directory: d:\Sutanu_WorkSpace\PruningNAS\PruningNAS


In [ ]:

# Initialize the model
basedir=''
path='./dataset/cifar10'
select_model='MobilenetV2'
pruning_type='CP'
#model_path='./checkpoint/vgg_mrl_99.51375579833984.pth'
model_path=r'.\checkpoint\MobilenetV2\MobilenetV2_cifar_95.029999.pth'
# Load the saved state_dict correctly
model = torch.load(model_path, map_location=torch.device(device),weights_only=False)  # Use 'cpu' if necessary

model.to(device)

sparsity_dict  = {
"features.0.0": 0.40,
"features.1.block.0": 0.50,
"features.1.block.3": 0.60,
"features.2.block.0": 0.60,
"features.2.block.3": 0.60,
"features.2.block.6": 0.50,
"features.3.block.0": 0.80,
"features.3.block.3": 0.70,
"features.3.block.6": 0.70,
"features.4.block.0": 0.60,
"features.4.block.3": 0.30,
"features.4.block.6": 0.50,
"features.5.block.0": 0.80,
"features.5.block.3": 0.70,
"features.5.block.6": 0.80,
"features.6.block.0": 0.80,
"features.6.block.3": 0.40,
"features.6.block.6": 0.80,
"features.7.block.0": 0.60,
"features.7.block.3": 0.40,
"features.7.block.6": 0.60,
"features.8.block.0": 0.90,
"features.8.block.3": 0.90,
"features.8.block.6": 0.90,
"features.9.block.0": 0.90,
"features.9.block.3": 0.90,
"features.9.block.6": 0.90,
"features.10.block.0": 0.90,
"features.10.block.3": 0.90,
"features.10.block.6": 0.90,
"features.11.block.0": 0.40,
"features.11.block.3": 0.50,
"features.11.block.6": 0.40,
"features.12.block.0": 0.90,
"features.12.block.3": 0.90,
"features.12.block.6": 0.90,
"features.13.block.0": 0.90,
"features.13.block.3": 0.90,
"features.13.block.6": 0.90,
"features.14.block.0": 0.50,
"features.14.block.3": 0.50,
"features.14.block.6": 0.70,
"features.15.block.0": 0.90,
"features.15.block.3": 0.70,
"features.15.block.6": 0.90,
"features.16.block.0": 0.80,
"features.16.block.3": 0.70,
"features.16.block.6": 0.80,
"features.17.block.0": 0.90,
"features.17.block.3": 0.90,
"features.17.block.6": 0.90,
"features.18": 0.90,
"fc": 0.90
}


# {
# "model.0": 0.1,
# "model.3.depthwise": 0.2,
# "model.4.depthwise": 0.1,
# "model.5.depthwise": 0.1,
# "model.6.depthwise": 0.2,
# "model.7.depthwise": 0.3,
# "model.8.depthwise": 0.2,
# "model.9.depthwise": 0.3,
# "model.10.depthwise": 0.2,
# "model.11.depthwise": 0.3,
# "model.12.depthwise": 0.5,
# "model.13.depthwise": 0.4,
# "model.14.depthwise": 0.7,
# "model.15.depthwise": 0.8
# }


Define experimental params here:

In [4]:
num_finetune_epochs = 300
lr=0.01

In [5]:
train_dataloader,test_dataloader=get_dataloaders(path, batch_size=256 ) # Basemodel
dense_model_accuracy=evaluate(model,test_dataloader)
print('dense_model_accuracy:',dense_model_accuracy)


d:\Sutanu_WorkSpace\PruningNAS\.venv\Lib\site-packages\torchvision\datasets\cifar.py:83: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  entry = pickle.load(f, encoding="latin1")
                                                     

dense_model_accuracy: (95.0199966430664, 0.0)


In [6]:
pruned_model=copy.deepcopy(model)
pruned_model.requires_grad_(False)  # Disable gradients before pruning

if pruning_type=='FGP':
    isCallback=True
    pruner = FineGrainedPruner(pruned_model, sparsity_dict)
elif pruning_type=='CP':
    pruned_model=ChannelPrunner(pruned_model, sparsity_dict,select_model)
    pruner=None
    isCallback=False
else:
    print('pruning_type doesn\'t exists')
    exit

print_model(pruned_model)
print(f'The sparsity of each layer becomes')
for name, param in pruned_model.named_parameters():
    print(f'  {name}: {get_sparsity(param):.2f}')


features.0.0.weight torch.Size([29, 3, 3, 3])
features.0.1.weight torch.Size([29])
features.0.1.bias torch.Size([29])
features.1.block.0.weight torch.Size([29, 1, 3, 3])
features.1.block.1.weight torch.Size([29])
features.1.block.1.bias torch.Size([29])
features.1.block.3.weight torch.Size([13, 29, 1, 1])
features.1.block.4.weight torch.Size([13])
features.1.block.4.bias torch.Size([13])
features.2.block.0.weight torch.Size([86, 13, 1, 1])
features.2.block.1.weight torch.Size([86])
features.2.block.1.bias torch.Size([86])
features.2.block.3.weight torch.Size([86, 1, 3, 3])
features.2.block.4.weight torch.Size([86])
features.2.block.4.bias torch.Size([86])
features.2.block.6.weight torch.Size([19, 86, 1, 1])
features.2.block.7.weight torch.Size([19])
features.2.block.7.bias torch.Size([19])
features.3.block.0.weight torch.Size([115, 19, 1, 1])
features.3.block.1.weight torch.Size([115])
features.3.block.1.bias torch.Size([115])
features.3.block.3.weight torch.Size([115, 1, 3, 3])
featur

In [ ]:
# model_path=r'.\checkpoint\Densenet-121\CP\Densenet-121_cifar_CP_94.58999633789062.pth'
# # Load the saved state_dict correctly
# pruned_model = torch.load(model_path, map_location=torch.device(device),weights_only=False)  # Use 'cpu' if necessary

# pruned_model.to(device)

In [7]:

dense_model_size = get_model_size(model, count_nonzero_only=True)
sparse_model_size = get_model_size(pruned_model, count_nonzero_only=True)

print(f"Sparse model has size={sparse_model_size:.2f} MiB = {sparse_model_size / dense_model_size * 100:.2f}% of dense model size")
sparse_model_accuracy,_ = evaluate(pruned_model, test_dataloader)
print(f"Sparse model has accuracy={sparse_model_accuracy:.2f}% before fintuning")

# Re-enable gradients for training
pruned_model.requires_grad_(True)

lr
optimizer = torch.optim.SGD(pruned_model.parameters(), lr=lr, momentum=0.9, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, num_finetune_epochs)
criterion = nn.CrossEntropyLoss()


Sparse model has size=2.77 MiB = 32.42% of dense model size


Sparse model has accuracy=10.00% before fintuning


In [8]:
pruned_model_accuracy,best_pruned_model,accuracies,train_losses,test_losses=TrainingPrunned(pruned_model,train_dataloader,test_dataloader,criterion, optimizer, pruner,scheduler=scheduler,num_finetune_epochs=num_finetune_epochs,isCallback=isCallback)
print(pruned_model_accuracy)


Finetuning Fine-grained Pruned Sparse Model


    Epoch 1 Test accuracy:80.41% / Best Accuracy: 80.41%, train loss: 0.6766, test loss: 0.6091


    Epoch 2 Test accuracy:84.85% / Best Accuracy: 84.85%, train loss: 0.3272, test loss: 0.4541


    Epoch 3 Test accuracy:84.72% / Best Accuracy: 84.85%, train loss: 0.2635, test loss: 0.4752


    Epoch 4 Test accuracy:86.75% / Best Accuracy: 86.75%, train loss: 0.2335, test loss: 0.4143


...

    Epoch 296 Test accuracy:94.62% / Best Accuracy: 94.70%, train loss: 0.0002, test loss: 0.2927


    Epoch 297 Test accuracy:94.54% / Best Accuracy: 94.70%, train loss: 0.0002, test loss: 0.2951


    Epoch 298 Test accuracy:94.62% / Best Accuracy: 94.70%, train loss: 0.0002, test loss: 0.2938


    Epoch 299 Test accuracy:94.63% / Best Accuracy: 94.70%, train loss: 0.0002, test loss: 0.2934


    Epoch 300 Test accuracy:94.62% / Best Accuracy: 94.70%, train loss: 0.0002, test loss: 0.2901
94.69999694824219


In [12]:
print(pruned_model_accuracy)
basedir='.'

torch.save(best_pruned_model, f'{basedir}/checkpoint/{select_model}/{pruning_type}/{select_model}_cifar_{pruning_type}_{pruned_model_accuracy}.pth')

titel_append=f'of {pruning_type} based Pruned {select_model.title()} model'
save_path=f'{basedir}/checkpoint/{select_model}/{pruning_type}/{select_model}_cifar_{pruning_type}'

plot_accuracy(accuracies,titel_append=titel_append,save_path=save_path+'_acc.png' )
plot_loss(train_losses,test_losses,titel_append=titel_append,save_path=save_path+'_loss.png')

94.56999969482422
[94.54999542236328, 94.56999969482422, 94.43999481201172, 94.25, 91.60999298095703, 90.66999816894531, 91.8899917602539, 92.43999481201172, 92.0999984741211, 91.45999908447266, 91.89999389648438, 90.63999938964844, 92.45999908447266, 92.43999481201172, 92.12999725341797, 92.65999603271484, 91.39999389648438, 92.93999481201172, 92.02999114990234, 92.32999420166016, 91.93999481201172, 92.15999603271484, 92.47999572753906, 92.14999389648438, 92.4699935913086, 92.20999145507812, 92.33999633789062, 92.5999984741211, 92.35999298095703, 92.57999420166016, 92.41999816894531, 92.35999298095703, 92.72999572753906, 92.60999298095703, 92.37999725341797, 93.15999603271484, 92.93999481201172, 92.5199966430664, 93.39999389648438, 92.79999542236328, 92.73999786376953, 93.06999969482422, 93.12999725341797, 93.06999969482422, 93.16999816894531, 93.18999481201172, 92.65999603271484, 92.52999877929688, 93.05999755859375, 92.9699935913086, 92.88999938964844, 92.37999725341797, 92.60999298

In [ ]:
best_pruned_model=pruned_model
best_pruned_model.cuda()

In [11]:

num_finetune_epochs=300
optimizer = torch.optim.SGD(best_pruned_model.parameters(), lr=0.01, momentum=0.9, weight_decay=1e-4)

pruned_model_accuracy,best_pruned_model,accuracies,train_losses,test_losses=TrainingPrunned(best_pruned_model,train_dataloader,test_dataloader,criterion, optimizer, pruner,scheduler=scheduler,num_finetune_epochs=num_finetune_epochs,isCallback=isCallback)


Finetuning Fine-grained Pruned Sparse Model


    Epoch 1 Test accuracy:94.55% / Best Accuracy: 94.55%, train loss: 0.0002, test loss: 0.2959


    Epoch 2 Test accuracy:94.57% / Best Accuracy: 94.57%, train loss: 0.0002, test loss: 0.2954


    Epoch 3 Test accuracy:94.44% / Best Accuracy: 94.57%, train loss: 0.0002, test loss: 0.2951


    Epoch 4 Test accuracy:94.25% / Best Accuracy: 94.57%, train loss: 0.0003, test loss: 0.3163


...

    Epoch 296 Test accuracy:94.40% / Best Accuracy: 94.57%, train loss: 0.0002, test loss: 0.2958


    Epoch 297 Test accuracy:94.43% / Best Accuracy: 94.57%, train loss: 0.0002, test loss: 0.2966


    Epoch 298 Test accuracy:94.38% / Best Accuracy: 94.57%, train loss: 0.0002, test loss: 0.2944


    Epoch 299 Test accuracy:94.35% / Best Accuracy: 94.57%, train loss: 0.0002, test loss: 0.2966


    Epoch 300 Test accuracy:94.40% / Best Accuracy: 94.57%, train loss: 0.0002, test loss: 0.2945
